# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

> ⚠️ **Updated:** Original notebook loaded API keys from a `.env` file using `dotenv`. This updated version uses `getpass` so you can enter keys directly at runtime without needing a `.env` file.

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
#from getpass import getpass

_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

#LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")
LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT")

#OPENAI_API_KEY = getpass('Please enter your OpenAI API Key: ')
#LANGCHAIN_API_KEY = getpass('Please enter your LangChain API Key: ')
#HUGGINGFACEHUB_API_TOKEN = getpass('Please enter your HuggingFace Hub API Token: ')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

#os.environ['LANGCHAIN_API_KEY'] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]="my-lab2"

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
os.environ["LANGSMITH_ENDPOINT"] = LANGSMITH_ENDPOINT


In [2]:
#Importing Client from Langsmith
from langsmith import Client
#client = Client(api_key=LANGSMITH_API_KEY)

client = Client(
    api_key=LANGSMITH_API_KEY,
    api_url=LANGSMITH_ENDPOINT
)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('422 Client Error: unknown for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Unprocessable entity: invalid feedback part for feedback.d013194f-416b-44ea-828f-eb7f43f11979: score has a maximum precision of 4 decimal places, was 0.45193422160777186"}\n')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('422 Client Error: unknown for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Unprocessable entity: invalid feedback part for feedback.21e6ed02-a6a7-45c7-99e4-c30b894980bd: score has a maximum precision of 4 decimal places, was 0.3918205716263913"}\n')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in La

### Create Dataset


> ⚠️ **Updated:** Original used `ccdv/cnn_dailymail` with `version=` and `trust_remote_code=True` which are no longer supported. Updated to use `cnn_dailymail` with `name=` parameter.

In [3]:
from datasets import load_dataset
#cnn_dataset = load_dataset("cnn_dailymail", name="3.0.0")
cnn_dataset = load_dataset("xsum")

# error
#cnn_dataset = load_dataset(
#    "ccdv/cnn_dailymail", version
#    ="3.0.0",
#    trust_remote_code=True
#)

c:\Users\bianc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
#def add_prefix(example):
#    return {
#        **example,
#        "article": f"Summarize this news:\n{example['article']}"
#    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [5]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [6]:
cnn_dataset['train'][0]

{'document': 'The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining wall did not fail.\n"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally apprecia

In [7]:
#Get just a few news to test
MAX_NEWS=10
#sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS))

sample_cnn

Dataset({
    features: ['document', 'summary', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [8]:
print(sample_cnn[0])

{'document': 'Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.\nWorkers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.\nThe Welsh Government said more people than ever were getting help to address housing problems.\nChanges to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.\nPrison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.\nHowever, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.\nAndrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [9]:
import datetime

In [10]:
import uuid
#input_key=['article']
input_key=['document']
#output_key=['highlights']
output_key=['summary']

NAME_DATASET=f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
print(NAME_DATASET)

Summarize_dataset_2026-03-04 11:37:11


In [11]:
#import os
#print(os.environ.get("LANGCHAIN_ENDPOINT"))
#print("LANGSMITH_API_KEY:", LANGSMITH_API_KEY)
#print(client.list_datasets(limit=1))

#for ds in client.list_datasets(limit=1):
#    print(ds)

In [12]:
#This creates the dataset in LangSmith with the content in sample_cnn - If you run this more than once you will get POST errors
df = sample_cnn.to_pandas()

# Create dataset
dataset = client.create_dataset(
    dataset_name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
)

# Add examples
client.create_examples(
    inputs=[{"document": row["document"]} for _, row in df.iterrows()],
    outputs=[{"summary": row["summary"]} for _, row in df.iterrows()],
    dataset_id=dataset.id,
)
#client.create_examples(
#    inputs=[{"article": row["article"]} for _, row in df.iterrows()],
#    outputs=[{"highlights": row["highlights"]} for _, row in df.iterrows()],
#    dataset_id=dataset.id,
#)

# error using upload_dataframe
#dataset = client.upload_dataframe(
#    df=sample_cnn,
#    input_keys=input_key,
#    output_keys=output_key,
#    name=NAME_DATASET,
#    description="Test Embedding distance between model summarizations",
#    data_type="kv"
#)
print(f"✅ Dataset created: {NAME_DATASET}")

✅ Dataset created: Summarize_dataset_2026-03-04 11:37:11


In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

> ⚠️ **Updated:** Original used `HuggingFaceHub` from `langchain` which is deprecated, and `t5-base` / `flax-community/t5-base-cnn-dm` which are no longer accessible via the HuggingFace Inference API. Updated to use direct HTTP requests to the HuggingFace Router API with `facebook/bart-large-cnn` as the base model (a reliable summarization model) and kept `flax-community/t5-base-cnn-dm` as the fine-tuned model.

In [13]:
import requests
import numpy as np
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def hf_summarize(text, repo_id):
    API_URL = f'https://router.huggingface.co/hf-inference/models/{repo_id}'
    headers = {'Authorization': f'Bearer {HUGGINGFACEHUB_API_TOKEN}'}
    try:
        response = requests.post(API_URL, headers=headers, json={'inputs': text[:1024]})
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list):
                return result[0].get('summary_text') or result[0].get('generated_text', '')
            return str(result)
        return f'API Error: {response.status_code}'
    except Exception as e:
        return f'Error: {str(e)}'

# Base model: facebook/bart-large-cnn (replaces t5-base)
def summarizer_base(inputs):
    return hf_summarize(inputs, 'facebook/bart-large-cnn')

# Fine-tuned model: flax-community/t5-base-cnn-dm
def summarizer_finetuned(inputs):
    return hf_summarize(inputs, 'flax-community/t5-base-cnn-dm')

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.

> ⚠️ **Updated:** Original used `run_on_dataset` and `RunEvalConfig` from `langchain.smith` which have been removed in newer LangChain versions. Updated to use `langsmith.evaluate` with a custom `embedding_distance_evaluator` function that computes cosine distance using OpenAI embeddings.

In [14]:
from langsmith import evaluate

def get_embedding(text):
    if not text or not str(text).strip():
        return np.zeros(1536)
    response = openai_client.embeddings.create(
        input=str(text)[:8000],
        model='text-embedding-3-small'
    )
    return np.array(response.data[0].embedding)

def embedding_distance_evaluator(run, example):
    prediction = (run.outputs or {}).get('output', '')
    #reference = (example.outputs or {}).get('highlights', '')
    reference = (example.outputs or {}).get('summary', '')
    emb1 = get_embedding(str(prediction))
    emb2 = get_embedding(str(reference))
    cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-9)
    return {'key': 'embedding_distance', 'score': 1 - cosine_sim}

#We are using just one of the multiple evaluators available on LangSmith.
#string_distance is left as a comment in case you want to add a second evaluator.
#string_distance_evaluator can be added to the evaluators list similarly.

### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

> ⚠️ **Updated:** Original used `run_on_dataset` with `summarizer_base` (t5-base). Updated to use `langsmith.evaluate` with `facebook/bart-large-cnn` as the base model since t5-base is no longer accessible via the HuggingFace Inference API.

In [15]:
def summarize_bart(inputs: dict):
    #output = hf_summarize(inputs['article'], 'facebook/bart-large-cnn')
    output = hf_summarize(inputs['document'], 'facebook/bart-large-cnn')
    return {'output': output}

bart_results = evaluate(
    summarize_bart,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='BART-large-CNN',
    client=client,
)

View the evaluation results for experiment: 'BART-large-CNN-69e249aa' at:
https://eu.smith.langchain.com/o/ebda7115-89e9-4719-984a-61913bf186de/datasets/0a0f7031-f0f7-484d-b702-9925f54c1520/compare?selectedSessions=fa96e2d5-1b5e-4642-9e39-4402b060d585




0it [00:00, ?it/s]

> ⚠️ **Updated:** Original used `run_on_dataset` with `summarizer_finetuned`. Updated to use `langsmith.evaluate` with a direct HuggingFace API call.

In [16]:
def summarize_t5_finetuned(inputs: dict):
    #output = hf_summarize(inputs['article'], 'flax-community/t5-base-cnn-dm')
    output = hf_summarize(inputs['document'], 'flax-community/t5-base-cnn-dm')
    return {'output': output}

finetuned_t5_results = evaluate(
    summarize_t5_finetuned,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='T5-FineTuned',
    client=client,
)

View the evaluation results for experiment: 'T5-FineTuned-b49aee09' at:
https://eu.smith.langchain.com/o/ebda7115-89e9-4719-984a-61913bf186de/datasets/0a0f7031-f0f7-484d-b702-9925f54c1520/compare?selectedSessions=ed9b788c-fd5f-4e56-9847-a1f6926071ea




0it [00:00, ?it/s]

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

> ⚠️ **Updated:** Original used `from langchain.chat_models import ChatOpenAI` which is deprecated, and `run_on_dataset` which has been removed. Updated to use `from langchain_openai import ChatOpenAI` and `langsmith.evaluate`.

In [17]:
#import sys
#!{sys.executable} -m pip install langchain-openai

from langchain_openai import ChatOpenAI

open_aillm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.0, openai_api_key=OPENAI_API_KEY)

def summarize_openai(inputs: dict):
    #article = inputs['article'][:3000]
    article = inputs['document'][:3000]
    result = open_aillm.invoke(article)
    return {'output': result.content if result.content else 'No summary generated'}

openai_results = evaluate(
    summarize_openai,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='OpenAI-GPT35',
    client=client,
)

View the evaluation results for experiment: 'OpenAI-GPT35-fff02045' at:
https://eu.smith.langchain.com/o/ebda7115-89e9-4719-984a-61913bf186de/datasets/0a0f7031-f0f7-484d-b702-9925f54c1520/compare?selectedSessions=f8d1e7b4-9e59-4677-b885-59617d2cc4d7




0it [00:00, ?it/s]

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.